In [ ]:
import pickle
from pathlib import Path
import torch
import importlib

import tests.load
import tests.constrained_generate

importlib.reload(tests.load)
importlib.reload(tests.constrained_generate)

from tests.load import model, tokenizer, dataset
from tests.constrained_generate import generate_constrained

device = next(model.parameters()).device

idx = 0
row = dataset.df.iloc[idx]
unmut = row['unmutated_proteome']
mut = row['mutated_proteome']

changes = [(j, unmut[j], mut[j]) for j in range(min(len(unmut), len(mut))) if unmut[j] != mut[j]]
mutation_pos = [p for p, _, _ in changes]

phylo_idx = dataset.genome_to_phylo_idx[idx]
phylo_dists = torch.tensor(dataset.phylo_matrix[phylo_idx], dtype=torch.float, device=device).unsqueeze(0)

generated_cont, generated_tokens = generate_constrained(
    model=model,
    tokenizer=tokenizer,
    unmutated_proteome=unmut,
    mutation_positions=mutation_pos,
    target_length=len(mut) + 10000,
    max_new_tokens_per_window=2000,
    overlap_tokens=384,
    temperature=0.0,
    phylo_dists=phylo_dists
)

output = {
    'genome_id': row['genome_id'],
    'unmutated_proteome': unmut,
    'mutated_proteome': mut,
    'generated_proteome': generated_cont,
    'mutation_positions': mutation_pos,
}

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
out_path = out_dir / 'test1_generated.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(output, f)

print(f'Saved {out_path}')

Using device: mps
Loading tokenizer...
Loading model architecture...
Loading finetune checkpoint...
Model loaded successfully

Loading small dataset subset for inference testing...
Filtered to 382 genomes with reversions applied (mutated pairs)
Loading chunk: samples 0 to 5
Loading chunk cache from explicit path: /Users/rishi/Documents/phylogen/cache/ecoli_pairs_concat_mode_finetune_chunk1024_overlap512_mutated_only_True_start0_maxall.chunks.pkl
→ Loaded 4,524 cached chunks successfully
Dataset ready: 4524 chunks from 5 genomes

Encoding unmutated proteome...
Starting generation from position 1,551,635 (target length ~1,632,870)
Generating: 0/1,632,870 (0.0%)
Generating: 1,632,870/1,632,870 (100.00%)
Saved outputs/test1_generated.pkl


/Users/rishi/Documents/phylogen/tests/constrained_generate.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  unmut_ids = torch.tensor(
